# Speed Estimation Pipeline

This notebook integrates the complete speed estimation pipeline:
1. Vehicle detection (YOLOv9)
2. Lane detection (SimpleLaneDetector)
3. Vehicle tracking (ByteTrack)
4. Homography calculation from dashed lanes
5. Speed estimation using tripwires

**Pipeline Flow:**
```
For each frame:
  1. Detect vehicles → bounding boxes
  2. Detect lanes → polylines  
  3. Track vehicles → track IDs
  4. Calculate homography → bird's-eye view
  5. Measure lane spacing → meters-per-pixel
  6. Track vehicles crossing tripwires → calculate speed
```

## 1. Setup and Imports

In [ ]:
# Install dependencies
!pip install -q ultralytics opencv-python-headless pillow matplotlib numpy scipy lap

print("Dependencies installed!")

In [ ]:
import sys
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from collections import defaultdict, deque
from ultralytics import YOLO

# Add project root to path
project_root = r'C:\SpeedRadar'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Imports successful!")

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Load Trained Models

### 2.1 Load Vehicle Detection Model (YOLOv9)

In [ ]:
# Load YOLOv9 vehicle detection model
vehicle_model_path = r'C:\SpeedRadar\models\weights\yolov9_vehicle_detection_best.pt'

if os.path.exists(vehicle_model_path):
    vehicle_detector = YOLO(vehicle_model_path)
    print(f"Vehicle detector loaded from: {vehicle_model_path}")
else:
    print("WARNING: Vehicle detector not found. Using pretrained YOLO model.")
    vehicle_detector = YOLO('yolov8m.pt')
    print("Loaded pretrained YOLOv8m model instead.")

# Set model to evaluation mode
vehicle_detector.model.eval()
print("Vehicle detector ready!")

### 2.2 Load Lane Detection Model

In [ ]:
# Import lane detector architecture from notebook 03
from torchvision import models

class SimpleLaneDetector(nn.Module):
    def __init__(self, num_classes=1):
        super(SimpleLaneDetector, self).__init__()
        
        # Use ResNet18 as backbone
        resnet = models.resnet18(pretrained=False)
        
        # Encoder (ResNet backbone)
        self.layer0 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4
        
        # Decoder (upsampling)
        self.up1 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec1 = self._make_decoder_block(512, 256)
        
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = self._make_decoder_block(256, 128)
        
        self.up3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec3 = self._make_decoder_block(128, 64)
        
        self.up4 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec4 = self._make_decoder_block(64, 32)
        
        # Final convolution
        self.final = nn.Sequential(
            nn.Conv2d(32, num_classes, kernel_size=1),
            nn.Sigmoid()
        )
    
    def _make_decoder_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        # Encoder
        x0 = self.layer0(x)
        x1 = self.layer1(x0)
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        
        # Decoder with skip connections
        up1 = self.up1(x4)
        cat1 = torch.cat([up1, x3], dim=1)
        dec1 = self.dec1(cat1)
        
        up2 = self.up2(dec1)
        cat2 = torch.cat([up2, x2], dim=1)
        dec2 = self.dec2(cat2)
        
        up3 = self.up3(dec2)
        cat3 = torch.cat([up3, x1], dim=1)
        dec3 = self.dec3(cat3)
        
        up4 = self.up4(dec3)
        cat4 = torch.cat([up4, x0], dim=1)
        dec4 = self.dec4(cat4)
        
        # Final output
        out = self.final(dec4)
        
        # Upsample to original size
        out = nn.functional.interpolate(out, scale_factor=4, mode='bilinear', align_corners=False)
        
        return out

print("SimpleLaneDetector architecture loaded!")

In [ ]:
# Load lane detection model weights
lane_model_path = r'C:\SpeedRadar\models\weights\lane_detector_best.pt'

if os.path.exists(lane_model_path):
    lane_detector = SimpleLaneDetector(num_classes=1).to(device)
    lane_detector.load_state_dict(torch.load(lane_model_path, map_location=device))
    lane_detector.eval()
    print(f"Lane detector loaded from: {lane_model_path}")
else:
    print("WARNING: Lane detector not found. Will use dummy lane detection.")
    lane_detector = None

print("Lane detector ready!")

## 3. Homography and Scale Calculation Functions

### 3.1 Extract Lane Polylines from Segmentation Mask

In [ ]:
def extract_lane_polylines(mask, threshold=0.5, min_points=10):
    """
    Extract lane polylines from segmentation mask.
    
    Args:
        mask: Binary segmentation mask (H, W)
        threshold: Threshold for binarization
        min_points: Minimum number of points for a valid lane
    
    Returns:
        List of lane polylines, each as numpy array of (x, y) points
    """
    # Binarize mask
    binary_mask = (mask > threshold).astype(np.uint8) * 255
    
    # Find contours
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    lanes = []
    for contour in contours:
        # Approximate contour
        epsilon = 0.01 * cv2.arcLength(contour, False)
        approx = cv2.approxPolyDP(contour, epsilon, False)
        
        # Convert to polyline
        points = approx.squeeze()
        
        if len(points.shape) == 2 and len(points) >= min_points:
            # Sort by y-coordinate
            points = points[np.argsort(points[:, 1])]
            lanes.append(points)
    
    return lanes

print("Lane polyline extraction function defined.")

### 3.2 Homography Calculation (Adapted from existing notebook)

In [ ]:
def get_homography(original_image_pil, model_mask_640):
    """
    Calculate homography matrix to transform image to bird's-eye view.
    Adapted from car_and_road_recognition_model.ipynb
    """
    orig_w, orig_h = original_image_pil.size
    full_mask = cv2.resize(model_mask_640, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
    
    # Find all pixels of the mask
    all_y, all_x = np.where(full_mask > 0.5)
    
    if len(all_x) == 0 or len(all_y) == 0:
        return None, None
    
    # Calculate the range of x and y axis
    delta_x = np.max(all_x) - np.min(all_x)
    delta_y = np.max(all_y) - np.min(all_y)
    
    # If delta_x is significantly larger, we treat it as horizontal
    if delta_x > 1.2 * delta_y:
        # Horizontal roads, we take dots on the left and right side
        offset_x = int(delta_x * 0.1)
        left_x = np.min(all_x) + offset_x
        right_x = np.max(all_x) - offset_x
        
        # Edges of the road are on the top and bottom of that x
        left_indices = np.where(full_mask[:, left_x] > 0.5)[0]
        right_indices = np.where(full_mask[:, right_x] > 0.5)[0]
        
        if len(left_indices) == 0 or len(right_indices) == 0:
            return None, None
        
        # Ordering points for 400x800 vertical output
        src_pts = np.float32([
            [left_x, left_indices[0]],   
            [right_x, right_indices[0]], 
            [right_x, right_indices[-1]],
            [left_x, left_indices[-1]]   
        ])
    else:
        # Vertical road
        top_y = np.min(all_y) + 30
        bottom_y = np.max(all_y) - 20
        top_indices = np.where(full_mask[top_y, :] > 0.5)[0]
        bottom_indices = np.where(full_mask[bottom_y, :] > 0.5)[0]
        
        if len(top_indices) == 0 or len(bottom_indices) == 0:
            return None, None
        
        src_pts = np.float32([
            [top_indices[0], top_y], 
            [top_indices[-1], top_y],
            [bottom_indices[-1], bottom_y], 
            [bottom_indices[0], bottom_y]
        ])

    # Rotate the image to a bird's eye view
    dst_pts = np.float32([[0, 0], [400, 0], [400, 800], [0, 800]])
    M = cv2.getPerspectiveTransform(src_pts, dst_pts)
    warped = cv2.warpPerspective(np.array(original_image_pil), M, (400, 800))
    
    return warped, M

print("Homography calculation function defined.")

### 3.3 Extract Line Segments from Dashed Lanes

In [ ]:
def extract_line_segments(warped_img):
    """
    Extract line segments from warped bird's-eye view image.
    Adapted from car_and_road_recognition_model.ipynb
    """
    # Grayscale the image
    gray = cv2.cvtColor(warped_img, cv2.COLOR_RGB2GRAY)
    # Otsu threshold for extracting lines
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    height, width = thresh.shape
    
    # Find the best x axis by looking for dashed lines
    best_x = width // 2
    max_score = 0
    best_segments = []
    
    # Scan from left to right, staying away from sidewalk edges
    for x in range(50, width - 50, 10):
        # Create the tunnel around current x
        sample_width = 5
        tunnel = thresh[:, max(0, x - sample_width) : min(width, x + sample_width)]
        
        # Average of the tunnel to boolean
        line_map = np.mean(tunnel, axis=1) > 127
        
        # Group into segments
        segments = []
        if len(line_map) > 0:
            count = 1
            for i in range(1, len(line_map)):
                if line_map[i] == line_map[i-1]:
                    count += 1
                else:
                    segments.append((line_map[i-1], count))
                    count = 1
            segments.append((line_map[-1], count))
            
            # Score to avoid tram rails or long curbs
            transitions = len(segments)
            score = transitions
            
            for is_white, length in segments:
                # If the white line is too long (more than 40% of height), it's probably a rail
                if is_white and length > (height * 0.4):
                    score = 0
                    break
            
            # Take the x with best score
            if score > max_score:
                max_score = score
                best_x = x
                best_segments = segments
                
    if not best_segments: 
        return None, None, best_x
        
    # Find the first valid couple
    p_line, p_gap = None, None
    for i in range(len(best_segments) - 1):
        # Use a safe range for lines (30px to 300px)
        if best_segments[i][0] == True and 30 < best_segments[i][1] < 300:
            if best_segments[i+1][0] == False and 20 < best_segments[i+1][1] < 400:
                p_line = best_segments[i][1]
                p_gap = best_segments[i+1][1]
                break
                
    return p_line, p_gap, best_x

print("Line segment extraction function defined.")

### 3.4 Calculate Meters per Pixel

In [ ]:
def calculate_meters_per_pixel(p_line, p_gap, verbose=False):
    """
    Calculate scale factor (meters per pixel) from dashed lane markings.
    Adapted from car_and_road_recognition_model.ipynb
    """
    if p_line is None or p_gap is None:
        if verbose:
            print("System did not detect a clear line-gap pair.")
        return None
        
    ratio = p_line / p_gap
    
    # If the line is a lot bigger than the gap, we say it's 3m line and 1m gap
    if ratio >= 2.0:
        road_type = "City road (3m line, 1m gap)"
        meters_per_pixel = 3.0 / p_line
    else:
        road_type = "Open road (3m line, 3m gap)"
        meters_per_pixel = 3.0 / p_line
        
    # Safety check
    if meters_per_pixel > 0.5:
        if verbose:
            print("Warning: Scale factor is suspiciously large!")
        return None
        
    if verbose:
        print(f"Road type: {road_type}")
        print(f"Pixels: Line={p_line}px, Gap={p_gap}px (Ratio: {ratio:.2f})")
        print(f"Final scale: {meters_per_pixel:.5f} meters per pixel")
    
    return meters_per_pixel

print("Meters per pixel calculation function defined.")

## 4. Simple Tracking Implementation

### 4.1 Simple IoU-Based Tracker (Lightweight Alternative to ByteTrack)

In [ ]:
def calculate_iou(box1, box2):
    """
    Calculate Intersection over Union (IoU) between two boxes.
    Boxes in format [x1, y1, x2, y2]
    """
    x1_inter = max(box1[0], box2[0])
    y1_inter = max(box1[1], box2[1])
    x2_inter = min(box1[2], box2[2])
    y2_inter = min(box1[3], box2[3])
    
    if x2_inter < x1_inter or y2_inter < y1_inter:
        return 0.0
    
    inter_area = (x2_inter - x1_inter) * (y2_inter - y1_inter)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = box1_area + box2_area - inter_area
    
    return inter_area / union_area if union_area > 0 else 0.0

class SimpleTracker:
    """
    Simple IoU-based tracker for vehicles.
    """
    def __init__(self, iou_threshold=0.3, max_age=30):
        self.iou_threshold = iou_threshold
        self.max_age = max_age
        self.tracks = {}
        self.next_id = 1
        self.frame_count = 0
    
    def update(self, detections):
        """
        Update tracks with new detections.
        
        Args:
            detections: List of [x1, y1, x2, y2, confidence, class_id]
        
        Returns:
            List of [x1, y1, x2, y2, track_id, class_id]
        """
        self.frame_count += 1
        
        # Match detections to existing tracks
        matched_tracks = {}
        unmatched_detections = list(range(len(detections)))
        
        # Try to match each track
        for track_id, track_data in list(self.tracks.items()):
            best_iou = 0
            best_det_idx = -1
            
            for det_idx in unmatched_detections:
                det_box = detections[det_idx][:4]
                track_box = track_data['box']
                iou = calculate_iou(track_box, det_box)
                
                if iou > best_iou and iou > self.iou_threshold:
                    best_iou = iou
                    best_det_idx = det_idx
            
            if best_det_idx >= 0:
                # Match found
                det = detections[best_det_idx]
                self.tracks[track_id] = {
                    'box': det[:4],
                    'class_id': int(det[5]) if len(det) > 5 else 0,
                    'last_seen': self.frame_count
                }
                matched_tracks[track_id] = det[:4]
                unmatched_detections.remove(best_det_idx)
        
        # Create new tracks for unmatched detections
        for det_idx in unmatched_detections:
            det = detections[det_idx]
            track_id = self.next_id
            self.next_id += 1
            self.tracks[track_id] = {
                'box': det[:4],
                'class_id': int(det[5]) if len(det) > 5 else 0,
                'last_seen': self.frame_count
            }
            matched_tracks[track_id] = det[:4]
        
        # Remove old tracks
        to_remove = []
        for track_id, track_data in self.tracks.items():
            if self.frame_count - track_data['last_seen'] > self.max_age:
                to_remove.append(track_id)
        
        for track_id in to_remove:
            del self.tracks[track_id]
        
        # Return tracked objects
        tracked_objects = []
        for track_id, box in matched_tracks.items():
            class_id = self.tracks[track_id]['class_id']
            tracked_objects.append([*box, track_id, class_id])
        
        return tracked_objects

print("SimpleTracker defined!")

## 5. Speed Estimation Implementation

### 5.1 Speed Calculator with Tripwires

In [ ]:
class SpeedCalculator:
    """
    Calculate vehicle speeds using tripwire method.
    """
    def __init__(self, tripwire_y1=200, tripwire_y2=600, fps=30):
        """
        Args:
            tripwire_y1: First tripwire y-coordinate (in bird's-eye view)
            tripwire_y2: Second tripwire y-coordinate (in bird's-eye view)
            fps: Frames per second of video
        """
        self.tripwire_y1 = tripwire_y1
        self.tripwire_y2 = tripwire_y2
        self.fps = fps
        
        # Track crossings: {track_id: {'y1': frame, 'y2': frame}}
        self.crossings = defaultdict(dict)
        
        # Calculated speeds: {track_id: speed_kmh}
        self.speeds = {}
    
    def update(self, tracked_objects, homography_matrix, meters_per_pixel):
        """
        Update speed calculations with tracked objects.
        
        Args:
            tracked_objects: List of [x1, y1, x2, y2, track_id, class_id]
            homography_matrix: Homography matrix for bird's-eye transform
            meters_per_pixel: Scale factor (meters per pixel in bird's-eye view)
        
        Returns:
            Dictionary of {track_id: speed_kmh}
        """
        if homography_matrix is None or meters_per_pixel is None:
            return self.speeds
        
        current_frame = len(self.crossings)  # Simple frame counter
        
        for obj in tracked_objects:
            x1, y1, x2, y2, track_id, class_id = obj
            
            # Calculate center of bounding box
            center_x = (x1 + x2) / 2
            center_y = (y1 + y2) / 2
            
            # Transform center to bird's-eye view
            point = np.array([[[center_x, center_y]]], dtype=np.float32)
            transformed = cv2.perspectiveTransform(point, homography_matrix)
            bev_x, bev_y = transformed[0][0]
            
            # Check tripwire crossings
            if 'y1' not in self.crossings[track_id]:
                # Check if crossing first tripwire
                if abs(bev_y - self.tripwire_y1) < 20:  # Within 20px of tripwire
                    self.crossings[track_id]['y1'] = current_frame
            
            elif 'y2' not in self.crossings[track_id]:
                # Check if crossing second tripwire
                if abs(bev_y - self.tripwire_y2) < 20:  # Within 20px of tripwire
                    self.crossings[track_id]['y2'] = current_frame
                    
                    # Calculate speed
                    frame_diff = self.crossings[track_id]['y2'] - self.crossings[track_id]['y1']
                    if frame_diff > 0:
                        # Distance in pixels
                        pixel_distance = abs(self.tripwire_y2 - self.tripwire_y1)
                        
                        # Distance in meters
                        distance_meters = pixel_distance * meters_per_pixel
                        
                        # Time in seconds
                        time_seconds = frame_diff / self.fps
                        
                        # Speed in m/s
                        speed_ms = distance_meters / time_seconds
                        
                        # Convert to km/h
                        speed_kmh = speed_ms * 3.6
                        
                        # Sanity check (20-150 km/h)
                        if 20 <= speed_kmh <= 150:
                            self.speeds[track_id] = speed_kmh
        
        return self.speeds
    
    def get_speed(self, track_id):
        """Get speed for a specific track ID."""
        return self.speeds.get(track_id, None)

print("SpeedCalculator defined!")

## 6. Complete Pipeline Integration

In [ ]:
class SpeedEstimationPipeline:
    """
    Complete speed estimation pipeline integrating all components.
    """
    def __init__(self, vehicle_detector, lane_detector, fps=30):
        self.vehicle_detector = vehicle_detector
        self.lane_detector = lane_detector
        self.fps = fps
        
        self.tracker = SimpleTracker(iou_threshold=0.3, max_age=30)
        self.speed_calculator = SpeedCalculator(tripwire_y1=200, tripwire_y2=600, fps=fps)
        
        self.homography_matrix = None
        self.meters_per_pixel = None
    
    def process_frame(self, frame):
        """
        Process a single frame through the complete pipeline.
        
        Args:
            frame: Input image (numpy array, RGB)
        
        Returns:
            Dictionary with results:
            {
                'vehicles': [(x1, y1, x2, y2, track_id, class_id, speed_kmh), ...],
                'lanes': [lane_polylines],
                'speeds': {track_id: speed_kmh}
            }
        """
        h, w = frame.shape[:2]
        
        # 1. Detect vehicles
        vehicle_results = self.vehicle_detector.predict(frame, conf=0.25, iou=0.45, verbose=False)[0]
        detections = []
        
        for box in vehicle_results.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            conf = box.conf[0].cpu().numpy()
            cls = int(box.cls[0].cpu().numpy())
            detections.append([x1, y1, x2, y2, conf, cls])
        
        # 2. Detect lanes
        lanes = []
        lane_mask = None
        
        if self.lane_detector is not None:
            # Prepare image for lane detection
            from torchvision import transforms
            transform = transforms.Compose([
                transforms.ToPILImage(),
                transforms.Resize((640, 640)),
                transforms.ToTensor()
            ])
            
            image_tensor = transform(frame).unsqueeze(0).to(device)
            
            with torch.no_grad():
                lane_mask = self.lane_detector(image_tensor)
                lane_mask = lane_mask.squeeze().cpu().numpy()
            
            # Extract polylines from mask
            lanes = extract_lane_polylines(lane_mask, threshold=0.5, min_points=10)
        
        # 3. Track vehicles
        tracked_objects = self.tracker.update(detections)
        
        # 4. Calculate homography and scale (once per frame or when lanes change)
        if lane_mask is not None and len(lanes) > 0:
            try:
                pil_image = Image.fromarray(frame)
                warped, M = get_homography(pil_image, lane_mask)
                
                if warped is not None and M is not None:
                    self.homography_matrix = M
                    
                    # Extract line segments
                    p_line, p_gap, _ = extract_line_segments(warped)
                    
                    # Calculate scale
                    self.meters_per_pixel = calculate_meters_per_pixel(p_line, p_gap, verbose=False)
            except Exception as e:
                pass  # Keep using previous homography if calculation fails
        
        # 5. Calculate speeds
        speeds = self.speed_calculator.update(
            tracked_objects, 
            self.homography_matrix, 
            self.meters_per_pixel
        )
        
        # 6. Combine results
        vehicles_with_speed = []
        for obj in tracked_objects:
            x1, y1, x2, y2, track_id, class_id = obj
            speed_kmh = speeds.get(track_id, None)
            vehicles_with_speed.append((x1, y1, x2, y2, track_id, class_id, speed_kmh))
        
        return {
            'vehicles': vehicles_with_speed,
            'lanes': lanes,
            'speeds': speeds
        }

print("SpeedEstimationPipeline defined!")

## 7. Testing and Visualization

### 7.1 Create Test Video from Images

In [ ]:
# Create a simple test video from validation images
val_images_dir = Path(r'C:\SpeedRadar\dataset\val\images')
image_files = sorted(list(val_images_dir.glob('*.jpg')))[:100]  # Use first 100 images

print(f"Creating test video from {len(image_files)} images...")

# Read first image to get dimensions
first_img = cv2.imread(str(image_files[0]))
height, width = first_img.shape[:2]

# Create video writer
output_video_path = r'C:\SpeedRadar\test_video.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video_writer = cv2.VideoWriter(output_video_path, fourcc, 30, (width, height))

for img_path in image_files:
    img = cv2.imread(str(img_path))
    video_writer.write(img)

video_writer.release()
print(f"Test video created: {output_video_path}")

### 7.2 Process Test Video

In [ ]:
# Initialize pipeline
pipeline = SpeedEstimationPipeline(vehicle_detector, lane_detector, fps=30)

# Open test video
cap = cv2.VideoCapture(output_video_path)

# Process frames
frame_count = 0
all_speeds = []

print("Processing video...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # Convert BGR to RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Process frame
    results = pipeline.process_frame(frame_rgb)
    
    # Collect speeds
    for track_id, speed in results['speeds'].items():
        all_speeds.append({'track_id': track_id, 'speed_kmh': speed, 'frame': frame_count})
    
    frame_count += 1
    
    if frame_count % 10 == 0:
        print(f"Processed {frame_count} frames...")

cap.release()

print(f"\nProcessing complete!")
print(f"Total frames processed: {frame_count}")
print(f"Speeds calculated: {len(all_speeds)}")

### 7.3 Visualize Results on Sample Frame

In [ ]:
# Process and visualize a single frame
test_image_path = image_files[10]  # Use 10th image
test_image = cv2.imread(str(test_image_path))
test_image_rgb = cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB)

results = pipeline.process_frame(test_image_rgb)

# Visualize
fig, ax = plt.subplots(1, 1, figsize=(15, 10))
ax.imshow(test_image_rgb)

# Draw vehicles with speeds
colors = {0: 'red', 1: 'green', 2: 'blue'}
class_names = {0: 'car', 1: 'truck', 2: 'bus'}

for vehicle in results['vehicles']:
    x1, y1, x2, y2, track_id, class_id, speed_kmh = vehicle
    
    color = colors.get(class_id, 'yellow')
    class_name = class_names.get(class_id, 'vehicle')
    
    # Draw bounding box
    from matplotlib.patches import Rectangle
    rect = Rectangle(
        (x1, y1), x2-x1, y2-y1,
        linewidth=2,
        edgecolor=color,
        facecolor='none'
    )
    ax.add_patch(rect)
    
    # Draw label with speed
    label = f"ID:{track_id} {class_name}"
    if speed_kmh is not None:
        label += f" {speed_kmh:.1f}km/h"
    
    ax.text(
        x1, y1 - 5,
        label,
        color=color,
        fontsize=10,
        weight='bold',
        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2)
    )

# Draw lanes
for lane in results['lanes']:
    if len(lane) > 0:
        # Scale lane points back to original image size
        lane_scaled = lane * [test_image_rgb.shape[1]/640, test_image_rgb.shape[0]/640]
        ax.plot(lane_scaled[:, 0], lane_scaled[:, 1], 'y-', linewidth=3, alpha=0.5)

ax.set_title('Speed Estimation Results', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()

print(f"\nDetected {len(results['vehicles'])} vehicles")
print(f"Detected {len(results['lanes'])} lane markings")
print(f"Calculated {len(results['speeds'])} speeds")

## 8. Export Results

In [ ]:
import json

# Format results for export
export_data = []

for speed_data in all_speeds:
    export_data.append({
        "track_id": int(speed_data['track_id']),
        "speed_kmh": float(speed_data['speed_kmh']),
        "frame": int(speed_data['frame']),
        "confidence": 0.85  # Placeholder confidence
    })

# Save to JSON
output_json_path = r'C:\SpeedRadar\speed_estimations.json'
with open(output_json_path, 'w') as f:
    json.dump(export_data, f, indent=2)

print(f"Results exported to: {output_json_path}")
print(f"Total speed measurements: {len(export_data)}")

# Display sample results
if len(export_data) > 0:
    print("\nSample results:")
    for i, result in enumerate(export_data[:5]):
        print(f"  {i+1}. Track ID: {result['track_id']}, Speed: {result['speed_kmh']:.1f} km/h, Frame: {result['frame']}")

## 9. Statistics and Validation

In [ ]:
# Calculate statistics
if len(all_speeds) > 0:
    speeds_only = [s['speed_kmh'] for s in all_speeds]
    
    print("="*70)
    print("SPEED ESTIMATION STATISTICS")
    print("="*70)
    
    print(f"\nTotal measurements: {len(speeds_only)}")
    print(f"Average speed: {np.mean(speeds_only):.2f} km/h")
    print(f"Median speed: {np.median(speeds_only):.2f} km/h")
    print(f"Min speed: {np.min(speeds_only):.2f} km/h")
    print(f"Max speed: {np.max(speeds_only):.2f} km/h")
    print(f"Std deviation: {np.std(speeds_only):.2f} km/h")
    
    # Plot distribution
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.hist(speeds_only, bins=20, color='blue', alpha=0.7, edgecolor='black')
    plt.axvline(np.mean(speeds_only), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(speeds_only):.1f} km/h')
    plt.xlabel('Speed (km/h)')
    plt.ylabel('Frequency')
    plt.title('Speed Distribution')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.boxplot(speeds_only)
    plt.ylabel('Speed (km/h)')
    plt.title('Speed Box Plot')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Validation check
    print("\nVALIDATION:")
    if 30 <= np.mean(speeds_only) <= 80:
        print("  Average speed is within expected range for city streets (30-80 km/h)")
    else:
        print("  WARNING: Average speed is outside expected range. Check calibration.")
    
    print("\n" + "="*70)
else:
    print("No speed measurements were calculated.")
    print("This may indicate:")
    print("  1. Lane detection failed (no dashed lanes detected)")
    print("  2. Vehicles did not cross both tripwires")
    print("  3. Homography calculation failed")

## 10. Summary

In [ ]:
print("="*70)
print("SPEED ESTIMATION PIPELINE SUMMARY")
print("="*70)

print("\n1. PIPELINE COMPONENTS:")
print("   - Vehicle Detection: YOLOv9 (or YOLOv8)")
print("   - Lane Detection: U-Net with ResNet-18 backbone")
print("   - Vehicle Tracking: Simple IoU-based tracker")
print("   - Speed Calculation: Tripwire method with homography")

print("\n2. PROCESSING RESULTS:")
print(f"   - Total frames processed: {frame_count}")
print(f"   - Speed measurements: {len(all_speeds)}")
if len(all_speeds) > 0:
    print(f"   - Average speed: {np.mean([s['speed_kmh'] for s in all_speeds]):.2f} km/h")

print("\n3. OUTPUT FILES:")
print(f"   - Speed data: {output_json_path}")
print(f"   - Test video: {output_video_path}")

print("\n4. PIPELINE WORKFLOW:")
print("   Frame → Vehicle Detection → Lane Detection → Tracking")
print("   → Homography Calculation → Speed Estimation → Results")

print("\n5. NEXT STEPS FOR IMPROVEMENT:")
print("   - Integrate ByteTrack for better tracking performance")
print("   - Fine-tune lane detection on more samples")
print("   - Calibrate tripwire positions for specific camera setups")
print("   - Add confidence scoring for speed estimates")
print("   - Implement real-time video processing")

print("\n" + "="*70)